# 20 - Measure and Build Packed Recovered L78

Notebook operacional para transformar o marco `packed/materialized L78` em um artefato mensurável e reproduzível.

Objetivos:
- carregar `student_pruned -> mlp_ternary_2bit.bin -> adapter recovered L78` na mesma ordem validada no notebook 18;
- medir tamanho em disco, tempo de carga, RAM/VRAM e throughput simples de geração;
- gerar um pacote operacional local com manifesto, loader e relatório JSON;
- não fazer retreino e não alterar resultados anteriores.

Importante: este notebook mede usabilidade operacional. Ele não substitui uma avaliação maior de generalização.

In [ ]:
# Dependências principais. No Colab, rode esta célula se o ambiente ainda não tiver os pacotes.
import sys, subprocess, importlib.util

required = {
    'torch': 'torch',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'peft': 'peft',
    'safetensors': 'safetensors',
    'sentencepiece': 'sentencepiece',
    'psutil': 'psutil',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Dependências principais já disponíveis.')

In [ ]:
# Bootstrap Colab/local e paths oficiais.
from pathlib import Path
import os, sys, json, time, gc, hashlib, shutil, platform

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
PACKED_RECOVERY_RESULTS_DIR = RESULTS_DIR / 'packed_recovery'
BUILD_DIR = PROJECT_ROOT / 'build' / 'packed_recovered_l78'

for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, EXPORTS_DIR, PACKED_RECOVERY_RESULTS_DIR, BUILD_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
PRUNED_CHECKPOINT = CHECKPOINTS_DIR / 'student_pruned'
PACKED_RECOVERY_ADAPTER = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
QUESTIONS_PATH = DATA_DIR / 'gemma4_100_questions.json'
EXPECTED_PACKED_SHA256 = '0b6849fe3ceaab615a4e54f63f5a96e9d9d431ba03b7b791a7ab0638ea174116'

PACKAGE_CANDIDATES = [
    EXPORTS_DIR / 'ternary_mlp_l78',
    DEPLOY_DIR,
    PROJECT_ROOT / 'exports' / 'ternary_mlp_l78',
    PROJECT_ROOT / 'deploy',
]
PACKAGE_DIR = next((p for p in PACKAGE_CANDIDATES if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)
if PACKAGE_DIR is None:
    raise FileNotFoundError('Pacote packed não encontrado em: ' + json.dumps([str(p) for p in PACKAGE_CANDIDATES], ensure_ascii=False, indent=2))

MANIFEST_PATH = PACKAGE_DIR / 'manifest.json'
BIN_PATH = PACKAGE_DIR / 'mlp_ternary_2bit.bin'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('PACKAGE_DIR =', PACKAGE_DIR)
print('BUILD_DIR =', BUILD_DIR)
print('PRUNED_CHECKPOINT =', PRUNED_CHECKPOINT)
print('PACKED_RECOVERY_ADAPTER =', PACKED_RECOVERY_ADAPTER)

In [ ]:
# Utilitários de medição, hash, tamanho e memória.
import psutil
import numpy as np
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

process = psutil.Process(os.getpid())

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def dir_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return None
    if path.is_file():
        return path.stat().st_size
    total = 0
    for item in path.rglob('*'):
        if item.is_file():
            total += item.stat().st_size
    return total

def format_bytes(n):
    if n is None:
        return None
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def memory_snapshot(label):
    snap = {
        'label': label,
        'rss_bytes': process.memory_info().rss,
        'rss_human': format_bytes(process.memory_info().rss),
    }
    if torch.cuda.is_available():
        snap.update({
            'cuda_device': torch.cuda.get_device_name(0),
            'cuda_allocated_bytes': torch.cuda.memory_allocated(),
            'cuda_reserved_bytes': torch.cuda.memory_reserved(),
            'cuda_max_allocated_bytes': torch.cuda.max_memory_allocated(),
            'cuda_allocated_human': format_bytes(torch.cuda.memory_allocated()),
            'cuda_reserved_human': format_bytes(torch.cuda.memory_reserved()),
            'cuda_max_allocated_human': format_bytes(torch.cuda.max_memory_allocated()),
        })
    return snap

def timed(label, fn):
    cleanup_memory()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    before = memory_snapshot(label + ':before')
    start = time.perf_counter()
    value = fn()
    elapsed = time.perf_counter() - start
    after = memory_snapshot(label + ':after')
    return value, {'label': label, 'elapsed_sec': elapsed, 'before': before, 'after': after}

manifest = load_json(MANIFEST_PATH)
questions = load_json(QUESTIONS_PATH) if QUESTIONS_PATH.exists() else []
packed_sha256 = sha256_file(BIN_PATH)

assert manifest.get('format') == 'rpt_hybrid_ternary_mlp_lora_v1', manifest.get('format')
assert manifest['summary']['num_tensors'] == 78, manifest['summary']
assert packed_sha256 == EXPECTED_PACKED_SHA256, packed_sha256
assert PRUNED_CHECKPOINT.exists(), PRUNED_CHECKPOINT
assert PACKED_RECOVERY_ADAPTER.exists(), PACKED_RECOVERY_ADAPTER

sizes = {
    'student_pruned': {'path': str(PRUNED_CHECKPOINT), 'bytes': dir_size_bytes(PRUNED_CHECKPOINT)},
    'adapter_recovered_l78': {'path': str(PACKED_RECOVERY_ADAPTER), 'bytes': dir_size_bytes(PACKED_RECOVERY_ADAPTER)},
    'packed_bin': {'path': str(BIN_PATH), 'bytes': dir_size_bytes(BIN_PATH), 'sha256': packed_sha256},
    'packed_manifest': {'path': str(MANIFEST_PATH), 'bytes': dir_size_bytes(MANIFEST_PATH)},
}
for item in sizes.values():
    item['human'] = format_bytes(item['bytes'])

print(json.dumps(sizes, ensure_ascii=False, indent=2))

In [ ]:
# 2-bit unpack e materialização dos 78 MLPs no student_pruned.
def unpack_codes_from_blob(blob, n):
    raw = np.frombuffer(blob, dtype=np.uint8)
    codes = np.empty(raw.size * 4, dtype=np.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    return codes[:n]

def unpack_tensor_from_blob(blob, n, shape, scale, dtype=torch.float32):
    codes = unpack_codes_from_blob(blob, n)
    vals = np.zeros(n, dtype=np.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    return torch.from_numpy(vals).view(tuple(shape)).to(dtype=dtype)

def materialize_packed_mlp_into_model(model, manifest, bin_path):
    named_params = dict(model.named_parameters())
    missing_params = []
    meta_params = []
    applied = []
    with open(bin_path, 'rb') as f:
        for item in manifest['tensors']:
            name = item['name']
            param = named_params.get(name)
            if param is None:
                missing_params.append(name)
                continue
            if getattr(param, 'is_meta', False):
                meta_params.append(name)
                continue
            f.seek(item['offset'])
            blob = f.read(item['packed_bytes'])
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=torch.float32)
            if tuple(param.shape) != tuple(recon.shape):
                raise RuntimeError(f'Shape mismatch for {name}: param={tuple(param.shape)} recon={tuple(recon.shape)}')
            with torch.no_grad():
                param.copy_(recon.to(device=param.device, dtype=param.dtype))
            applied.append(name)
            del recon
    if missing_params or meta_params:
        raise RuntimeError(json.dumps({'missing_params': missing_params, 'meta_params': meta_params}, ensure_ascii=False, indent=2))
    if len(applied) != 78:
        raise RuntimeError(f'Expected 78 MLP tensors applied, got {len(applied)}')
    return applied

def load_packed_recovered_l78():
    load_steps = []
    tokenizer, step = timed('load_tokenizer', lambda: AutoTokenizer.from_pretrained(str(PRUNED_CHECKPOINT), token=HF_TOKEN))
    load_steps.append(step)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model, step = timed('load_student_pruned', lambda: AutoModelForCausalLM.from_pretrained(
        str(PRUNED_CHECKPOINT),
        dtype='auto',
        device_map='auto',
        token=HF_TOKEN,
    ))
    load_steps.append(step)

    applied, step = timed('materialize_packed_mlp', lambda: materialize_packed_mlp_into_model(base_model, manifest, BIN_PATH))
    load_steps.append(step)

    model, step = timed('load_recovered_adapter', lambda: PeftModel.from_pretrained(
        base_model,
        str(PACKED_RECOVERY_ADAPTER),
        token=HF_TOKEN,
        is_trainable=False,
    ))
    load_steps.append(step)
    model.eval()
    return model, tokenizer, applied, load_steps

In [ ]:
# Carga medida do artefato final.
cleanup_memory()
environment = {
    'platform': platform.platform(),
    'python': sys.version,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cuda_total_memory_human': format_bytes(torch.cuda.get_device_properties(0).total_memory) if torch.cuda.is_available() else None,
}
model, tokenizer, applied, load_steps = load_packed_recovered_l78()
print('applied_packed_mlp_tensors =', len(applied))
print(json.dumps({'environment': environment, 'load_steps': load_steps}, ensure_ascii=False, indent=2))

In [ ]:
# Throughput simples de geração. Não é benchmark científico; serve para saber se o artefato é utilizável.
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
MAX_NEW_TOKENS = 32
DEFAULT_PROMPTS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Quem desenvolveu a teoria da relatividade?',
    'Em Python, qual palavra-chave define uma função?',
    'Se hoje é segunda-feira, que dia será depois de 3 dias?',
]
prompts = [q.get('question', q) for q in questions[:5]] if questions else DEFAULT_PROMPTS

def generate_once(question):
    prompt = PROMPT_TEMPLATE.format(question=question)
    inputs = tokenizer(prompt, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start
    prompt_tokens = int(inputs['input_ids'].shape[-1])
    total_tokens = int(output.shape[-1])
    generated_tokens = max(0, total_tokens - prompt_tokens)
    text = tokenizer.decode(output[0][prompt_tokens:], skip_special_tokens=True).strip()
    return {
        'question': question,
        'answer': text,
        'prompt_tokens': prompt_tokens,
        'generated_tokens': generated_tokens,
        'elapsed_sec': elapsed,
        'tokens_per_sec': generated_tokens / elapsed if elapsed > 0 else None,
    }

generation_rows = []
for q in prompts:
    row = generate_once(q)
    generation_rows.append(row)
    print(json.dumps(row, ensure_ascii=False, indent=2))

valid_tps = [r['tokens_per_sec'] for r in generation_rows if r['tokens_per_sec'] is not None]
generation_summary = {
    'num_prompts': len(generation_rows),
    'total_generated_tokens': sum(r['generated_tokens'] for r in generation_rows),
    'total_elapsed_sec': sum(r['elapsed_sec'] for r in generation_rows),
    'avg_tokens_per_sec': sum(valid_tps) / len(valid_tps) if valid_tps else None,
}
print(json.dumps(generation_summary, ensure_ascii=False, indent=2))

In [ ]:
# Gera pacote operacional: manifesto de build, loader standalone e README local.
created_at = time.strftime('%Y-%m-%d %H:%M:%S')
result_payload = {
    'notebook': '20_measure_and_build_packed_recovered_l78',
    'purpose': 'Operational measurement and reproducible build metadata for packed recovered L78.',
    'created_at': created_at,
    'load_order': 'student_pruned -> materialize mlp_ternary_2bit.bin -> PeftModel.from_pretrained(recovered adapter)',
    'paths': {
        'project_root': str(PROJECT_ROOT),
        'student_pruned': str(PRUNED_CHECKPOINT),
        'adapter_recovered_l78': str(PACKED_RECOVERY_ADAPTER),
        'manifest': str(MANIFEST_PATH),
        'packed_bin': str(BIN_PATH),
        'build_dir': str(BUILD_DIR),
    },
    'packed_bin_sha256': packed_sha256,
    'applied_packed_mlp_tensors': len(applied),
    'sizes': sizes,
    'environment': environment,
    'load_steps': load_steps,
    'generation_summary': generation_summary,
    'generation_rows': generation_rows,
    'baselines': {
        'hf_gemma_2_2b_it_original_reference_accuracy': 0.90,
        'student_pruned_reference_accuracy': 0.92,
        'clean_base_plus_l78_reference_accuracy': 0.92,
        'packed_base_no_adapter_accuracy': 0.01,
        'old_l78_on_packed_accuracy': 0.63,
        'recovered_packed_l78_local_accuracy': 1.00,
    },
    'source_results': {
        'reproduced_eval': 'results/packed_recovery/18_test_packed_recovered_l78_20260422_205445.json',
        'recovery_report': 'results/packed_recovery/17_t4_resume_report.json',
    },
}

build_manifest_path = BUILD_DIR / 'packed_recovered_l78_manifest.json'
save_json(result_payload, build_manifest_path)

loader_source = '''\
"""Standalone loader for RPT packed recovered L78.

Usage:
    model, tokenizer, report = load_packed_recovered_l78('/path/to/gemma4_theory')
"""
from pathlib import Path
import json, hashlib
import numpy as np
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

EXPECTED_PACKED_SHA256 = '0b6849fe3ceaab615a4e54f63f5a96e9d9d431ba03b7b791a7ab0638ea174116'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def unpack_codes_from_blob(blob, n):
    raw = np.frombuffer(blob, dtype=np.uint8)
    codes = np.empty(raw.size * 4, dtype=np.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    return codes[:n]

def unpack_tensor_from_blob(blob, n, shape, scale, dtype=torch.float32):
    codes = unpack_codes_from_blob(blob, n)
    vals = np.zeros(n, dtype=np.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    return torch.from_numpy(vals).view(tuple(shape)).to(dtype=dtype)

def materialize_packed_mlp_into_model(model, manifest, bin_path):
    named_params = dict(model.named_parameters())
    applied = []
    with open(bin_path, 'rb') as f:
        for item in manifest['tensors']:
            name = item['name']
            param = named_params.get(name)
            if param is None:
                raise RuntimeError(f'Missing parameter: {name}')
            if getattr(param, 'is_meta', False):
                raise RuntimeError(f'Meta parameter cannot be materialized directly: {name}')
            f.seek(item['offset'])
            blob = f.read(item['packed_bytes'])
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=torch.float32)
            if tuple(param.shape) != tuple(recon.shape):
                raise RuntimeError(f'Shape mismatch for {name}: param={tuple(param.shape)} recon={tuple(recon.shape)}')
            with torch.no_grad():
                param.copy_(recon.to(device=param.device, dtype=param.dtype))
            applied.append(name)
    if len(applied) != 78:
        raise RuntimeError(f'Expected 78 tensors, got {len(applied)}')
    return applied

def resolve_package_dir(project_root):
    candidates = [project_root / 'exports' / 'ternary_mlp_l78', project_root / 'deploy']
    for path in candidates:
        if (path / 'manifest.json').exists() and (path / 'mlp_ternary_2bit.bin').exists():
            return path
    raise FileNotFoundError('Packed package not found under exports/ternary_mlp_l78 or deploy')

def load_packed_recovered_l78(project_root, token=None, device_map='auto'):
    project_root = Path(project_root)
    checkpoints = project_root / 'checkpoints'
    pruned_checkpoint = checkpoints / 'student_pruned'
    adapter = checkpoints / 'student_packed_ternary_recovered_l78_t4_resume'
    package_dir = resolve_package_dir(project_root)
    manifest_path = package_dir / 'manifest.json'
    bin_path = package_dir / 'mlp_ternary_2bit.bin'
    packed_sha = sha256_file(bin_path)
    if packed_sha != EXPECTED_PACKED_SHA256:
        raise RuntimeError(f'Packed SHA mismatch: {packed_sha}')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    tokenizer = AutoTokenizer.from_pretrained(str(pruned_checkpoint), token=token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base_model = AutoModelForCausalLM.from_pretrained(str(pruned_checkpoint), dtype='auto', device_map=device_map, token=token)
    applied = materialize_packed_mlp_into_model(base_model, manifest, bin_path)
    model = PeftModel.from_pretrained(base_model, str(adapter), token=token, is_trainable=False)
    model.eval()
    report = {'packed_sha256': packed_sha, 'applied_packed_mlp_tensors': len(applied), 'load_order': 'student_pruned -> packed MLP -> recovered adapter'}
    return model, tokenizer, report
'''
(BUILD_DIR / 'loader_packed_recovered_l78.py').write_text(loader_source, encoding='utf-8')

readme_text = f'''# Packed Recovered L78 Operational Build

Generated at: {created_at}

Load order:
1. `student_pruned`
2. materialize `mlp_ternary_2bit.bin` using `manifest.json`
3. apply `student_packed_ternary_recovered_l78_t4_resume`

Main files:
- `packed_recovered_l78_manifest.json`: measured build metadata
- `loader_packed_recovered_l78.py`: standalone loader

Validated packed SHA: `{packed_sha256}`
Applied packed MLP tensors: `{len(applied)}`

This build keeps the benchmark claim separate from operational measurements. Use notebook 18 for reproduced accuracy and this notebook for runtime/build data.
'''
(BUILD_DIR / 'README.md').write_text(readme_text, encoding='utf-8')

result_path = PACKED_RECOVERY_RESULTS_DIR / f"20_measure_and_build_packed_recovered_l78_{time.strftime('%Y%m%d_%H%M%S')}.json"
save_json(result_payload, result_path)

print('build_manifest:', build_manifest_path)
print('loader:', BUILD_DIR / 'loader_packed_recovered_l78.py')
print('readme:', BUILD_DIR / 'README.md')
print('result:', result_path)

## Como interpretar o notebook 20

- Se a carga falhar por `meta parameter`, rode em ambiente com memória suficiente ou ajuste `device_map` para não deixar os MLPs materializados em meta tensors.
- Se o SHA falhar, o binário packed não é o mesmo validado no marco. Não continue sem investigar.
- O throughput de 5 prompts é apenas um smoke test operacional. Para claim de qualidade, usar avaliações maiores e inéditas.
- O pacote em `build/packed_recovered_l78` é derivado localmente e pode ser regenerado; o estado científico fica nos JSONs versionados em `results/packed_recovery`.